# Canvasly Support Operations — Exploratory Data Analysis, Anomaly Detection & Capacity Model

This notebook tells the quantitative story behind Canvasly's support crisis (`N = 30` ticket sample + operational baseline metrics). Rather than accepting aggregate metrics at face value (`6 min` median FRT, `14 min` AHT, `71%` CSAT), we decompose the dataset by customer segment, channel, agent behavior, time of day, and recorded vs. actual wall-clock wait times to uncover why Enterprise churn has more than doubled from `1.8%` to `4.2%` per quarter — and why the two-week manual triage pilot failed to stop two major Enterprise accounts from churning.

In [1]:
import csv, statistics, collections
import numpy as np
import matplotlib.pyplot as plt

with open('/home/alloyce/sol/Copy of canvasly_techops_case_data - canvasly_techops_case_data (1).csv') as f:
    rows = list(csv.DictReader(f))

print(f'Total rows: {len(rows)} | Total columns: {len(rows[0])}')
print('Schema & Missing/NULL Check:')
for col in rows[0].keys():
    nulls = sum(1 for r in rows if r[col] is None or str(r[col]).strip() == '' or str(r[col]).strip().lower() in ('null', 'na', 'nan', 'none'))
    dtype = 'int' if col in ('first_response_min', 'resolution_min', 'csat_score') else 'str'
    print(f'  {col:25s}: type={dtype:5s} | missing/null={nulls} ({nulls/len(rows)*100:.1f}%)')

Total rows: 30 | Total columns: 11
Schema & Missing/NULL Check:
  ticket_id                : type=str   | missing/null=0 (0.0%)
  created_at               : type=str   | missing/null=0 (0.0%)
  channel                  : type=str   | missing/null=0 (0.0%)
  category                 : type=str   | missing/null=0 (0.0%)
  priority_assigned        : type=str   | missing/null=0 (0.0%)
  first_response_min       : type=int   | missing/null=0 (0.0%)
  resolution_min           : type=int   | missing/null=0 (0.0%)
  agent_name               : type=str   | missing/null=0 (0.0%)
  escalated                : type=str   | missing/null=0 (0.0%)
  csat_score               : type=int   | missing/null=0 (0.0%)
  agent_internal_notes     : type=str   | missing/null=0 (0.0%)


### Analysis of Schema & Missing/NULL Values

1. **Explicit Completeness vs. Structural Schema Gaps:** All 30 rows and 11 columns contain `0` explicit `NULL` or empty values (`0.0%` missingness), so no rows should be dropped or imputed. However, the most important finding in the schema is **what is structurally missing from Zendesk's structured fields**:
   - There is **no `account_tier`**, **`customer_arr`**, **`seat_count`**, or **`organization_id`** column in Zendesk.
   - Instead, critical Enterprise context (`Meridian Health, 450 seats`, `NovaTech, 500 seats`, `CoreVista, 300 seats`, `Ridgeline Corp, 250 seats`, `Luma Analytics, 200 seats`, `DataForge, 175 seats`, plus an 80-seat Enterprise onboarding in `T-4805`) is buried solely inside unstructured free-text `agent_internal_notes` *after* an agent manually looks up the customer in Salesforce and the Admin Portal.
2. **Timestamp Granularity:** `created_at` spans a 35-hour operational window from `2026-01-02 09:14` to `2026-01-03 20:00` EST, allowing us to evaluate both daytime (`08:00–18:00` EST) and evening/overnight (`18:00–08:00` EST) arrival patterns.

In [2]:
for col in ['first_response_min', 'resolution_min', 'csat_score']:
    vals = np.array([float(r[col]) for r in rows])
    q25, med, q75, p90 = np.percentile(vals, [25, 50, 75, 90])
    iqr = q75 - q25
    upper_fence = q75 + 1.5 * iqr
    outliers = [(r['ticket_id'], float(r[col]), r['channel'], r['category']) for r in rows if float(r[col]) > upper_fence]
    print(f'{col:20s} | min={vals.min():5.1f} Q25={q25:5.1f} med={med:5.1f} Q75={q75:5.1f} P90={p90:5.1f} max={vals.max():5.1f} | mean={vals.mean():6.2f} std={vals.std():6.2f} IQR={iqr:5.1f} | UpperFence={upper_fence:5.1f}')
    print(f'  IQR Outliers: {outliers}')

first_response_min   | min=  1.0 Q25=  3.0 med=  5.0 Q75= 31.0 P90= 52.1 max=412.0 | mean= 29.37 std= 74.27 IQR= 28.0 | UpperFence= 73.0
  IQR Outliers: [('T-4815', 412.0, 'email', 'Billing - Invoice Question'), ('T-4828', 88.0, 'email', 'Account - Cancellation')]
resolution_min       | min=  7.0 Q25= 17.2 med= 36.0 Q75= 85.8 P90=187.8 max=485.0 | mean= 78.13 std=105.43 IQR= 68.5 | UpperFence=188.5
  IQR Outliers: [('T-4815', 485.0, 'email', 'Billing - Invoice Question'), ('T-4823', 340.0, 'email', 'Account - Enterprise Onboarding'), ('T-4829', 195.0, 'chat', 'Account - Access')]
csat_score           | min=  1.0 Q25=  2.0 med=  4.0 Q75=  5.0 P90=  5.0 max=  5.0 | mean=  3.40 std=  1.40 IQR=  3.0 | UpperFence=  9.5
  IQR Outliers: []


### Analysis of Target Distributions & Statistical Outliers

- **Extreme Right-Skew in `first_response_min`:** While the median FRT is `5.0 min` (closely matching the company's `6.0 min` baseline), the mean is `29.37 min` (`5.9x` the median) with a 90th percentile of `52.1 min`. The two statistical upper-fence outliers (`> 73.0 min`) belong to **the exact same Enterprise account (`DataForge, 175 seats`)**:
  - `T-4815` (`412 min` / `6.9 hours` FRT): An overnight email billing question where the customer followed up twice before anyone opened the ticket.
  - `T-4828` (`88 min` FRT): DataForge's formal **cancellation request** submitted the very next afternoon at `16:30` EST, which was tagged `Low` priority and sat untouched for nearly an hour and a half.
- **Resolution Time Outliers Are 100% Enterprise Accounts:** All three statistical outliers in `resolution_min` (`> 188.5 min`) are high-value Enterprise tickets: `T-4815` (`DataForge`, `485 min`), `T-4823` (`NovaTech, 500 seats`, `340 min` logged across a 2-day chaotic onboarding), and `T-4829` (`Ridgeline Corp, 250 seats`, `195 min` logged during an evening SSO lockout).

In [3]:
ent_ids = {'T-4804', 'T-4805', 'T-4810', 'T-4813', 'T-4815', 'T-4823', 'T-4824', 'T-4828', 'T-4829'}
ent = [r for r in rows if r['ticket_id'] in ent_ids]
non_ent = [r for r in rows if r['ticket_id'] not in ent_ids]

for name, grp in [('Confirmed Enterprise (N=9)', ent), ('Non-Enterprise / General (N=21)', non_ent)]:
    frt = [int(r['first_response_min']) for r in grp]
    res = [int(r['resolution_min']) for r in grp]
    csat = [int(r['csat_score']) for r in grp]
    csat_t2b = sum(1 for c in csat if c >= 4) / len(csat) * 100
    print(f'{name}:')
    print(f'  Channels: { {ch: sum(1 for r in grp if r["channel"]==ch) for ch in ["chat","email","voice"]} }')
    print(f'  Priority: { {p: sum(1 for r in grp if r["priority_assigned"]==p) for p in ["Low","Medium","High"]} }')
    print(f'  FRT:  median={statistics.median(frt):.1f}m, mean={statistics.mean(frt):.1f}m, min={min(frt)}m, max={max(frt)}m')
    print(f'  Res:  median={statistics.median(res):.1f}m, mean={statistics.mean(res):.1f}m')
    print(f'  CSAT: mean={statistics.mean(csat):.2f}, Top-2-Box (>=4)={csat_t2b:.1f}%')

Confirmed Enterprise (N=9):
  Channels: {'chat': 4, 'email': 5, 'voice': 0}
  Priority: {'Low': 4, 'Medium': 1, 'High': 4}
  FRT:  median=41.0m, mean=80.0m, min=4m, max=412m
  Res:  median=142.0m, mean=179.9m
  CSAT: mean=1.78, Top-2-Box (>=4)=11.1%
Non-Enterprise / General (N=21):
  Channels: {'chat': 16, 'email': 2, 'voice': 3}
  Priority: {'Low': 13, 'Medium': 6, 'High': 2}
  FRT:  median=4.0m, mean=7.7m, min=1m, max=47m
  Res:  median=22.0m, mean=34.5m
  CSAT: mean=4.10, Top-2-Box (>=4)=71.4%


### Analysis of the Segment Bifurcation (Enterprise vs. Non-Enterprise)

This segmentation reveals the central analytical insight of the case study:
- **Simpson's Paradox in CSAT & FRT:** Non-Enterprise tickets (`N = 21`) achieve a **Top-2-Box CSAT (`>= 4`) of `71.4%`** (mean `4.10 / 5.0`) and a **median FRT of `4.0 min`**, virtually identical to Canvasly's reported company-wide numbers (`71%` CSAT, `6 min` median FRT). High volumes of trivial Non-Enterprise chats (`Magic Import - Error` retry macros, feature requests) mask a total collapse in Enterprise support, where **Confirmed Enterprise tickets (`N = 9`) suffer a `1.78 / 5.0` average CSAT (`11.1%` Top-2-Box)**, a **`41.0 min` median FRT (`10.3x` slower)**, and a **`142.0 min` median reported resolution time (`6.5x` slower)**.
- **Channel & Priority Inversion:** `55.6%` (`5/9`) of Enterprise tickets arrive via **Email** (compared to only `9.5%` of Non-Enterprise tickets), and `44.4%` (`4/9`) of Enterprise tickets — including Luma Analytics' downgrade (`T-4810`), DataForge's cancellation (`T-4828`), and Meridian Health's Miro churn threat (`T-4824`) — are tagged **`Low` priority**.

In [4]:
print('=== ANOMALY DETECTION: REPORTED ZENDESK TIMERS VS. ACTUAL WALL-CLOCK WAIT ===')
anomalies = [
    ('T-4804', 6, 34, 67*60, 'Meridian Health (450 seats) routed to 67h Curation queue; gave CSAT=4 on chat handoff!'),
    ('T-4811', 4, 41, 67*60, 'Quoted 2-3h SLA while actual Design Curation queue is 67h.'),
    ('T-4816', 3, 38, 70*60, 'Quoted 2-3h SLA while actual Design Curation queue grew to 70+h.'),
    ('T-4824', 4, 45, 70*60, 'Meridian Health (450 seats) 2nd failure in 2 days; threatens switch to Miro by Friday.'),
    ('T-4805', 22, 67, 22+210, 'Eng Slack took 3.5h (210m) to respond, yet ZD resolution_min only recorded 67m.'),
    ('T-4813', 34, 187, 795, 'Created 19:45, picked up 9:00am next day (13.25h+), yet ZD logged 187m.'),
    ('T-4829', 41, 195, 885, 'Created 18:15 (3:15pm PST!), left overnight until 9am, yet ZD logged 195m.'),
    ('T-4823', 62, 340, 48*60, 'NovaTech (500 seats) onboarding spanned 2 days across 4 Slack channels, ZD logged 340m.')
]
for tid, frt, zres, actual, desc in anomalies:
    print(f'{tid} | FRT={frt:3d}m | ZD_Res={zres:3d}m | Actual_Wait={actual:4d}m ({actual/60:4.1f}h) | Distortion={actual/zres:5.1f}x | {desc}')

=== ANOMALY DETECTION: REPORTED ZENDESK TIMERS VS. ACTUAL WALL-CLOCK WAIT ===
T-4804 | FRT=  6m | ZD_Res= 34m | Actual_Wait=4020m (67.0h) | Distortion=118.2x | Meridian Health (450 seats) routed to 67h Curation queue; gave CSAT=4 on chat handoff!
T-4811 | FRT=  4m | ZD_Res= 41m | Actual_Wait=4020m (67.0h) | Distortion= 98.0x | Quoted 2-3h SLA while actual Design Curation queue is 67h.
T-4816 | FRT=  3m | ZD_Res= 38m | Actual_Wait=4200m (70.0h) | Distortion=110.5x | Quoted 2-3h SLA while actual Design Curation queue grew to 70+h.
T-4824 | FRT=  4m | ZD_Res= 45m | Actual_Wait=4200m (70.0h) | Distortion= 93.3x | Meridian Health (450 seats) 2nd failure in 2 days; threatens switch to Miro by Friday.
T-4805 | FRT= 22m | ZD_Res= 67m | Actual_Wait= 232m ( 3.9h) | Distortion=  3.5x | Eng Slack took 3.5h (210m) to respond, yet ZD resolution_min only recorded 67m.
T-4813 | FRT= 34m | ZD_Res=187m | Actual_Wait= 795m (13.2h) | Distortion=  4.3x | Created 19:45, picked up 9:00am next day (13.25h+), 

### Analysis of Timer Anomalies & Why the 2-Week Pilot Failed

Cross-referencing `resolution_min` against `agent_internal_notes` uncovers why two major Enterprise accounts still churned during the 2-week manual triage pilot despite `3 min` daytime FRT and `88%` CSAT:
1. **Handoff Gaming & Premature Closure (`93x–118x` clock distortion):** On `Magic Import - Quality` tickets (`T-4804, T-4811, T-4816, T-4824`), agents respond in `3–6 min`, quote a fictional `2–3 hour` SLA, route the file to the **Design Curation** backlog (**`67 to 70+ hours` wait**), and close the Zendesk ticket in `34–45 min`. In `T-4804`, Meridian Health (`450 seats`) even gave the agent a `CSAT = 4` for the polite 6-minute chat handoff — and then 26 hours later (`T-4824`) threatened to switch to **Miro** because their PDFs were stuck in Design Curation for the third time that week.
2. **Paused / Business-Hours-Only Timers (`3.5x–8.5x` clock distortion):** In `T-4805`, Engineering took `3.5 hours` (`210 min`) to reply in Slack, yet `resolution_min` recorded only `67 min`. In `T-4813` and `T-4829`, evening SSO lockouts (`6:15pm` and `7:45pm` EST — **which is `3:15pm` and `4:45pm` PST during West Coast business hours!**) sat unstaffed by Tier 2 overnight (`13–15 hours` actual outage), yet recorded `187–195 min`.
3. **In short:** The 2-week pilot optimized **time-to-hello (`FRT`)** while leaving **time-to-outcome** (Design Curation's 70h queue, Slack escalation black holes, Email backlogs, and West Coast/after-hours SSO lockouts) completely untouched.

### Conclusion & Executive Summary

1. **Root Cause Diagnosis:** Canvasly does not have a frontline chat speed problem; it has an **Enterprise outcome & routing blindness crisis** masked by high-volume Non-Enterprise chats.
2. **Capacity Unlock (`144 agent-hours/day` = `~22 FTEs` recovered):** Eliminating `5.5 min` of avoidable 4-system lookup time via a custom AWS-backed Zendesk Sidebar App (`-110 hrs/day`) and deflecting/auto-retrying `Magic Import - Error` transient timeouts (`26.7%` of sample volume; `-34 hrs/day`) reduces daily handle hours from `280 hrs/day` to `136 hrs/day` (`-51.4%`).
3. **Execution Sequence:**
   - **Week 1 (Stop the Bleeding, `<$400/mo`):** Sync Salesforce/Admin tier & seat metadata into Zendesk Organizations via AWS Lambda; deploy native Zendesk Omnichannel Routing + Enterprise VIP Priority (`P1` SSO/Access, `P2` Billing/Onboarding, `P0` Churn/Downgrade intercept); ship the read-only **Unified Context Sidebar App** (`Salesforce + Admin Portal + Policy/Pricing Sheet`); enforce **PagerDuty/Opsgenie Tier-2/Eng on-call paging** + **3pm–10pm EST West Coast shift stagger**; and priority-tag Enterprise jobs in the **Design Curation queue** while updating agent SLA quotes to live queue depth.
   - **Weeks 2–4 (Eliminate Preventable Load & Attrition):** Deploy automated retry/50MB pre-flight deflection for `Magic Import - Error`, launch structured Enterprise Onboarding Runbooks (`SSO + Custom Domain + Data Residency`) to support `500` new Enterprise accounts (`~3–4 onboardings/day`), replace broken Zapier/Google Sheets automations with native Zendesk Webhooks + Explore/S3, and eliminate manual cherry-picking (`Maria S.` vs. `James K.`) to cut the `11%/mo` agent attrition.